In [35]:
import pandas as pd
import numpy as np
import pandapipes as pp
from pandapipes.properties.fluids import create_constant_fluid
import os

# Network Creation

In [36]:
# --- network input data ---
mdot_consumers = 553 / 3600 #kg/s
delta_t_consumers = 30      # K
t_supply_producer = 70      # °C
t_ext_ground = 10           # °C

offset=(0.5,0.5,0.15)

# --- initialization ---
t_initial = 20              # °C
p_initial = 1               # bar


def create_net(nodes_file, pipes_file, fluid_props=None):
    """
    Thermal network creation in pandapipes

    :param nodes_file: path file CSV nodi
    :param pipes_file: path file CSV pipe

    :return: pandapipes network, node_map
    """

    # --- READ DATA ---
    nodes = pd.read_csv(nodes_file, sep=';')
    pipes = pd.read_csv(pipes_file, sep=';')

    # --- Heat transfer coefficient calculation dfrom input data ---
    def calculate_u_w_per_m2k(row):
        d_int = row['diameter_m']
        t_int = row['t_pipe_m']
        t_ins = row['t_ins_m']
        t_ext = 0.0

        lambda_int = 0.35   # W/mK
        lambda_ins = 0.026  # W/mK
        lambda_c = 0.4      # W/mK

        r_int = d_int / 2
        r_1 = r_int + t_int
        r_2 = r_1 + t_ins
        r_ext = r_2 + t_ext

        R_prime_int = np.log(r_1 / r_int) / (2 * np.pi * lambda_int)
        R_prime_ins = np.log(r_2 / r_1) / (2 * np.pi * lambda_ins)
        R_prime_c = np.log(r_ext / r_2) / (2 * np.pi * lambda_c)

        R_prime_global = R_prime_int + R_prime_ins + R_prime_c

        U_L = 1 / R_prime_global
        U = U_L / (2 * np.pi * r_int)

        return U

    pipes["u_w_per_m2k"] = pipes.apply(calculate_u_w_per_m2k, axis=1)

    # --- CREATE NETWORK ---
    net = pp.create_empty_network(name="Network_0")

    # --- DEFINE FLUID ---
    if fluid_props is None:
        fluid_props = dict(
            name="water_50",
            fluid_type="liquid",
            density=988,
            viscosity=0.0005434,
            heat_capacity=4180,
            thermal_conductivity=0.64,
            temperature=50 + 273.15
        )

    fluid = create_constant_fluid(**fluid_props)
    net.fluid = fluid

    # --- NODES ---
    node_map = {}
    for idx, row in nodes.iterrows():
        name = row["node_id"]
        name_s = f"{name}_s"
        name_r = f"{name}_r"

        # supply junction
        j_s = pp.create_junction(
            net,
            pn_bar=p_initial,
            tfluid_k=t_initial + 273.15,
            height_m=row.get("z",0),
            name=name_s,
            geodata=(row["x"], row["y"])
        )
        # return junction
        j_r = pp.create_junction(
            net,
            pn_bar=p_initial,
            tfluid_k=t_initial + 273.15,
            height_m=row.get("z",0),
            name=name_r,
            geodata=(row["x"] + offset[0], row["y"] + offset[1])
        )

        node_map[name] = {"supply": j_s, "return": j_r}

        # --- SUBSTATIONS ---
        if "SimpleDistrict" in name:
            pp.create_heat_consumer(
                net,
                from_junction=j_s,
                to_junction=j_r,
                deltat_k=delta_t_consumers,
                controlled_mdot_kg_per_s=mdot_consumers,
                name=name
            )

        # --- HEATING STATION ---
        if name == "i":
            pp.create_circ_pump_const_pressure(
                net,
                return_junction=j_r,
                flow_junction=j_s,
                p_flow_bar=p_initial,
                plift_bar=0.10,
                t_flow_k=t_supply_producer + 273.15,
                name="Producer"
            )

    # --- PIPES ---
    for idx, row in pipes.iterrows():
        start = row["start_node"]
        end = row["end_node"]
        name_s = f"{start}_to_{end}_s"
        name_r = f"{start}_to_{end}_r"

        start_s = node_map[start]["supply"]
        end_s   = node_map[end]["supply"]
        start_r = node_map[end]["return"]
        end_r   = node_map[start]["return"]

        # supply
        pp.create_pipe_from_parameters(
            net,
            from_junction=start_s,
            to_junction=end_s,
            length_km=row["length_m"]/1000,
            diameter_m=row["diameter_m"],
            k_mm=0.007,
            u_w_per_m2k=row["u_w_per_m2k"],
            text_k=t_ext_ground + 273.15,
            name=name_s
        )

        # return
        pp.create_pipe_from_parameters(
            net,
            from_junction=start_r,
            to_junction=end_r,
            length_km=row["length_m"]/1000,
            diameter_m=row["diameter_m"],
            k_mm=0.007,
            u_w_per_m2k=row["u_w_per_m2k"],
            text_k=t_ext_ground + 273.15,
            name=name_r
        )

    return net, node_map

# Load network data

In [37]:
nodes_file = "InputData/nodes_data.csv"
pipes_file = "InputData/pipes_data.csv"

net, node_map = create_net(nodes_file, pipes_file)

print(net)

This pandapipes network includes the following parameter tables:
   - junction (50 elements)
   - junction_geodata (50 elements)
   - pipe (48 elements)
   - heat_consumer (16 elements)
   - circ_pump_pressure (1 elements).
It contains the following fluid: 
Fluid water_50 (liquid) with properties:
   - density (Constant)
   - viscosity (Constant)
   - heat_capacity (Constant)
   - thermal_conductivity (Constant)
   - temperature (Constant)
and uses the following component models:
   - Junction
   - Pipe
   - ExtGrid
   - HeatConsumer
   - CirculationPumpPressure


In [38]:
import time

# --- To time execution calculation ---
start_time = time.perf_counter()

pp.pipeflow(net, mode="bidirectional", friction_model="colebrook")
end_time = time.perf_counter()
sim_time_total = end_time - start_time

print(f"Average simulation time: {sim_time_total:.2f} s")

Average simulation time: 0.01 s


In [40]:
def extract_kpis(net, filename=None):
    """
    KPIs extraction and saving:
    Time + 18 KPI.
    """

    res_p = net.res_pipe
    res_j = net.res_junction
    pipes = net.pipe
    juncs = net.junction

    #  UTILITIES
    def j_id(name):
        idx = juncs.index[juncs.name == name].tolist()
        if not idx:
            raise KeyError(f"Missing junction: {name}")
        return idx[0]

    def p_id(name):
        idx = pipes.index[pipes.name == name].tolist()
        if not idx:
            raise KeyError(f"Missing pipe: {name}")
        return idx[0]

    #  MASS FLOW
    mdot_supply = net.res_circ_pump_pressure.at[0, "mdot_from_kg_per_s"] * 3600

    #  PRESSURE DROP
    dp_i_to_e_s = (res_j.at[j_id("i_s"), "p_bar"] - res_j.at[j_id("e_s"), "p_bar"]) * 1e5
    dp_a_to_i_r = (res_j.at[j_id("a_r"), "p_bar"] - res_j.at[j_id("i_r"), "p_bar"]) * 1e5
    dp_i_to_h_r = (res_j.at[j_id("i_r"), "p_bar"] - res_j.at[j_id("h_r"), "p_bar"]) * 1e5

    # TEMPERATURE
    node_names = ["i", "h", "g", "f", "e", "SimpleDistrict_1"]
    node_list = [f"{n}_{side}" for side in ["s", "r"] for n in node_names]
    node_temps = [res_j.at[j_id(n), "t_k"] - 273.15 for n in node_list]

    # HEAT LOSSES
    # cp * mdot * (Tin − Tout)
    def q_loss(pipe_name):
        cp_50 = 4180 # water @ 50°C
        idx = p_id(pipe_name)
        mdot = res_p.at[idx, "mdot_from_kg_per_s"]
        Tin  = res_p.at[idx, "t_from_k"]
        Tout = res_p.at[idx, "t_to_k"]
        return mdot * cp_50 * (Tin - Tout)
    i_to_h_q_w = q_loss("i_to_h_s")

    # TOTAL HEAT SUPPLIED
    total_q_w = net.res_heat_consumer["qext_w"].sum()

    # BUILD RESULTS
    columns = [
        "Mass flow rate supply i (kg/h)",
        "Pressure drop supply between i and e (Pa)",
        "Pressure drop return between a and i (Pa)",
        "Pressure drop return between i and h (Pa)",
        "Fluid temperature supply i (°C)",
        "Fluid temperature supply h (°C)",
        "Fluid temperature supply g (°C)",
        "Fluid temperature supply f (°C)",
        "Fluid temperature supply e (°C)",
        "Fluid temperature supply SimpleDistrict_1 (°C)",
        "Fluid temperature return i (°C)",
        "Fluid temperature return h (°C)",
        "Fluid temperature return g (°C)",
        "Fluid temperature return f (°C)",
        "Fluid temperature return e (°C)",
        "Fluid temperature return SimpleDistrict_1 (°C)",
        "Heat loss supply between i and h (W)",
        "Total heat load supplied by heat source (W)",
    ]

    values = [
        mdot_supply,
        abs(dp_i_to_e_s),
        abs(dp_a_to_i_r),
        abs(dp_i_to_h_r),
        *node_temps,
        abs(i_to_h_q_w),
        abs(total_q_w),
    ]

    df = pd.DataFrame([values] * 10, columns=columns)
    df.insert(0, "Time", np.arange(10))

    # CSV SAVING (automatic file name)
    base_path = os.getcwd()
    results_folder = os.path.join(base_path, 'Results')

    if filename is None:
        filename = "network_CE0_pandapipes_results.csv"

    results_file = os.path.join(results_folder, filename)

    df.to_csv(results_file, index=False)
    print(f"Results for CE0 implemented in pandapipes saved in: {results_file}")

    return df


results_df = extract_kpis(net)

Results for CE0 implemented in pandapipes saved in: C:\Users\sara.ferrero\PycharmProjects\OSMSES_2026_Ferrero\Results\network_CE0_pandapipes_results.csv
